In [1]:
# ===============================
# 1. LIBRERÍAS
# ===============================
import pandas as pd
import numpy as np
import statsmodels.api as sm



In [5]:
# ===============================
# 2. CARGA DE DATOS
# ===============================
ruta = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos consolidados\Datos procesados\datos_procesados.xlsx"
df = pd.read_excel(ruta)
# Asegúrate de que esté ordenado temporalmente
df = df.sort_values(by=df.columns[0])  # cambia si tienes columna fecha específica


In [6]:
# =========================
# VARIABLES
# =========================

# Variable dependiente (ajústala a tu nombre real)
target = 'casos'  

# Variables meteorológicas locales
vars_meteo = [
    'hum_esp_bc', 'hum_rel_bc', 'temp_mean_bc', 'temp_max_bc',
    'vel_vi_mean_bc', 'vel_vi_max_bc', 'precip_bc', 'dias_lluvia_bc'
]

# Variables globales
vars_global = ['soi', 'sst']



In [8]:
# =========================
# CREACIÓN DE REZAGOS
# =========================

df_lags = df.copy()

# Meteorológicas: lags 1 a 8
for var in vars_meteo:
    for lag in range(1, 9):
        alias = {
            'temp_mean_bc': 'temp_bc',
            'vel_vi_mean_bc': 'vel_vi_bc',
            'precip_bc': 'prec',
            'dias_lluvia_bc': 'dias_lluvia'
        }
        source_var = alias.get(var, var)
        if source_var in df.columns:
            df_lags[f"{var}_lag_{lag}"] = df[source_var].shift(lag)
        else:
            df_lags[f"{var}_lag_{lag}"] = np.nan

# Globales: lags 8 a 12
for var in vars_global:
    for lag in range(8, 13):
        df_lags[f"{var}_lag_{lag}"] = df[var].shift(lag)

# Eliminar NA generados por rezagos
df_lags = df_lags.dropna()



In [16]:
# =========================
# TEST DE SIGNIFICANCIA
# =========================
from scipy.stats import spearmanr
resultados = []

for col in df_lags.columns:
    if "lag" in col:
        # Asegurar que exista la variable objetivo; intentar alternativas si no
        if target in df_lags.columns:
            y = df_lags[target]
        elif "casos_dengue" in df_lags.columns:
            y = df_lags["casos_dengue"]
        elif "casos_ln" in df_lags.columns:
            y = df_lags["casos_ln"]
        else:
            raise KeyError(f"Target '{target}' not found in df_lags columns")

        # Emparejar y eliminar NA antes de calcular Spearman
        pair = pd.concat([df_lags[col], y], axis=1).dropna()
        if pair.shape[0] < 2:
            corr, p_value = np.nan, np.nan
        else:
            corr, p_value = spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
        
        resultados.append({
            "variable": col,
            "correlacion": corr,
            "p_value": p_value
        })

resultados_df = pd.DataFrame(resultados)



In [17]:
# =========================
# FILTRADO DE SIGNIFICANCIA
# =========================

# Nivel de significancia (puedes ajustar a 0.01 si quieres ser más estricto)
alpha = 0.05

significativos = resultados_df[resultados_df['p_value'] < alpha]

# Ordenar por mayor correlación absoluta
significativos = significativos.reindex(
    significativos['correlacion'].abs().sort_values(ascending=False).index
)



In [18]:
# =========================
# RESULTADOS
# =========================

print("\n🔍 Rezagos significativos:")
print(significativos)

# Guardar resultados si quieres
significativos.to_excel("rezagos_significativos.xlsx", index=False)


🔍 Rezagos significativos:
              variable  correlacion       p_value
5     hum_esp_bc_lag_6     0.552946  2.431354e-21
7     hum_esp_bc_lag_8     0.547042  7.758871e-21
3     hum_esp_bc_lag_4     0.543817  1.448535e-20
2     hum_esp_bc_lag_3     0.542106  2.011511e-20
6     hum_esp_bc_lag_7     0.541472  2.270878e-20
..                 ...          ...           ...
66          soi_lag_10    -0.240578  1.262762e-04
65           soi_lag_9    -0.240491  1.270126e-04
17  temp_mean_bc_lag_2    -0.237571  1.541639e-04
16  temp_mean_bc_lag_1    -0.227284  2.993368e-04
18  temp_mean_bc_lag_3    -0.226801  3.085891e-04

[74 rows x 3 columns]
